# Adult Income — Logistic Regression LDS Benchmark (CPU)

Compares **Traceprop-LL** vs **Traceprop-BM** vs **Influence Functions (manual)** vs **Random** on the UCI Adult dataset.

**Why logistic regression?**
- No BatchNorm — per-sample last-layer gradient = full gradient (exact, not approximated)
- Retraining 500 subsets takes ~3–5 min on CPU (logistic regression is fast)
- Clean testbed to isolate attribution quality from architecture effects

**Runtime:** CPU is fine. Expected: ~10–15 minutes total.

Results saved to `tabular_logistic_lds.json`.

In [1]:
!pip install -q scikit-learn scipy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import spearmanr
import json
import time
import os

# CPU is fine for logistic regression
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

Device: cpu


## 0. Mount Google Drive (checkpoints)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

CKPT_DIR = '/content/drive/MyDrive/traceprop_tabular_ckpts'
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Checkpoint dir: {CKPT_DIR}')

def ckpt_exists(name):
    return os.path.exists(f'{CKPT_DIR}/{name}')

def save_ckpt(name, **kwargs):
    torch.save(kwargs, f'{CKPT_DIR}/{name}')
    print(f'  [Saved: {name}]')

def load_ckpt(name):
    data = torch.load(f'{CKPT_DIR}/{name}', weights_only=False)
    print(f'  [Loaded: {name}]')
    return data

Mounted at /content/drive
Checkpoint dir: /content/drive/MyDrive/traceprop_tabular_ckpts


## 1. Adult Income Dataset

In [3]:
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pandas as pd

print('Fetching Adult dataset...')
adult = fetch_openml('adult', version=2, as_frame=True, parser='auto')
X_raw = adult.data
y_raw = (adult.target == '>50K').astype(int)

# Convert categoricals to string first (avoids Categorical fillna issues)
X_enc = X_raw.copy()
for col in X_enc.select_dtypes(include='category').columns:
    X_enc[col] = X_enc[col].astype(str)

# Fill missing values (now safe — all object/numeric)
X_enc = X_enc.fillna('missing')

# One-hot encode string columns, scale numerics
X_dummies = pd.get_dummies(X_enc, drop_first=True).astype(float)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_dummies.values)
y_np = y_raw.values.astype(int)

# Use 6000 train / 1500 test for fast retraining
N_TRAIN, N_TEST = 6000, 1500
X_tr, X_te, y_tr, y_te = train_test_split(
    X_scaled, y_np, train_size=N_TRAIN, test_size=N_TEST,
    random_state=SEED, stratify=y_np
)

X_train = torch.tensor(X_tr, dtype=torch.float32).to(DEVICE)
y_train = torch.tensor(y_tr, dtype=torch.long).to(DEVICE)
X_test  = torch.tensor(X_te, dtype=torch.float32).to(DEVICE)
y_test  = torch.tensor(y_te, dtype=torch.long).to(DEVICE)

N_FEATURES = X_train.shape[1]
print(f'Adult: {N_TRAIN} train, {N_TEST} test, {N_FEATURES} features')
print(f'Label balance (train): {y_train.float().mean():.3f}')

Fetching Adult dataset...
Adult: 6000 train, 1500 test, 100 features
Label balance (train): 0.239


## 2. Logistic Regression Model

In [4]:
class LogisticRegression(nn.Module):
    def __init__(self, n_features, n_classes=2):
        super().__init__()
        self.linear = nn.Linear(n_features, n_classes)

    def forward(self, x):
        return self.linear(x)


def train_logreg(X, y, epochs=100, lr=0.1, weight_decay=1e-4, batch_size=256):
    model = LogisticRegression(X.shape[1]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()
    loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True)
    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()
    return model


@torch.no_grad()
def get_accuracy(model, X, y):
    model.eval()
    return (model(X).argmax(1) == y).float().mean().item()


@torch.no_grad()
def get_margin(model, X, y):
    """Output margin: logit[correct] - logit[incorrect]."""
    model.eval()
    logits = model(X)  # (N, 2)
    correct_logit = logits.gather(1, y.unsqueeze(1)).squeeze(1)
    wrong_logit = logits.sum(1) - correct_logit
    return (correct_logit - wrong_logit).cpu().numpy()


print(f'Logistic regression: {N_FEATURES}→2, params = {N_FEATURES * 2 + 2:,}')

Logistic regression: 100→2, params = 202


## 3. Train Full Model

In [5]:
if ckpt_exists('logreg_model.pt'):
    ckpt = load_ckpt('logreg_model.pt')
    model = LogisticRegression(N_FEATURES).to(DEVICE)
    model.load_state_dict(ckpt['state_dict'])
    full_acc = ckpt['full_acc']
    train_time = ckpt['train_time']
else:
    torch.manual_seed(SEED)
    t0 = time.perf_counter()
    model = train_logreg(X_train, y_train)
    train_time = time.perf_counter() - t0
    full_acc = get_accuracy(model, X_test, y_test)
    save_ckpt('logreg_model.pt', state_dict=model.state_dict(),
              full_acc=full_acc, train_time=train_time)

print(f'Test accuracy: {full_acc:.4f} (trained in {train_time:.2f}s)')

  [Loaded: logreg_model.pt]
Test accuracy: 0.8013 (trained in 21.28s)


## 4. Attribution Methods

For logistic regression, the last-layer gradient **is** the full gradient:
- `∂CE/∂W = (softmax(xW) - one_hot(y)) ⊗ x` — exact per-sample, no approximation
- Traceprop-LL ≡ Influence Functions for linear models
- Traceprop-BM averages gradients within each batch → loses intra-batch signal

In [6]:
# Config
N_SUBSETS = 500
SUBSET_RATIO = 0.5
BATCH_SIZE = 256
GRAD_DIM = N_FEATURES * 2  # logistic regression: W shape (2, n_features)

print(f'Gradient dim (full model = last layer): {GRAD_DIM}')


def per_sample_gradients(model, X, y):
    """Exact per-sample gradients for logistic regression.
    Uses activation ⊗ output_grad identity (same as last-layer trick).
    Returns (N, GRAD_DIM) float32 matrix.
    """
    model.eval()
    grads = []
    loader = DataLoader(TensorDataset(X, y), batch_size=BATCH_SIZE, shuffle=False)
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb)                      # (B, 2)
            sm = F.softmax(logits, dim=1)           # (B, 2)
            oh = F.one_hot(yb, num_classes=2).float()
            output_grad = sm - oh                   # (B, 2) — ∂CE/∂logits
            # ∂CE/∂W = output_grad_i ⊗ x_i → (B, 2*n_features)
            g = torch.einsum('bi,bj->bij', output_grad, xb).reshape(xb.shape[0], -1)
            grads.append(g.cpu())
    return torch.cat(grads, dim=0).numpy()


def batch_mean_gradients(model, X, y):
    """Batch-mean gradients (Traceprop-BM): one vector per batch, shared by all samples.
    Returns (N, GRAD_DIM) float32 matrix — same value repeated within each batch.
    """
    model.eval()
    result = np.zeros((len(X), GRAD_DIM), dtype=np.float32)
    loader = DataLoader(TensorDataset(X, y), batch_size=BATCH_SIZE, shuffle=False)
    idx = 0
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb)
            sm = F.softmax(logits, dim=1)
            oh = F.one_hot(yb, num_classes=2).float()
            output_grad = (sm - oh).mean(0, keepdim=True)   # (1, 2) — batch mean
            xb_mean = xb.mean(0, keepdim=True)              # (1, n_features)
            g_mean = torch.einsum('bi,bj->bij', output_grad, xb_mean).reshape(1, -1)
            bs = xb.shape[0]
            result[idx:idx+bs] = g_mean.cpu().numpy()       # same for all in batch
            idx += bs
    return result


print('Attribution functions defined.')

Gradient dim (full model = last layer): 200
Attribution functions defined.


## 5. Compute Attribution Matrices

In [7]:
# ── Traceprop-LL (per-sample last-layer = full gradient for linear model) ────
if ckpt_exists('tp_ll_tabular.pt'):
    ckpt = load_ckpt('tp_ll_tabular.pt')
    tp_ll_influence = ckpt['influence']
    tp_ll_time = ckpt['time_s']
    print(f'Traceprop-LL loaded: {tp_ll_influence.shape}')
else:
    t0 = time.perf_counter()
    train_grads_ll = per_sample_gradients(model, X_train, y_train)  # (N_TRAIN, GRAD_DIM)
    test_grads_ll  = per_sample_gradients(model, X_test,  y_test)   # (N_TEST, GRAD_DIM)
    tp_ll_influence = test_grads_ll @ train_grads_ll.T               # (N_TEST, N_TRAIN)
    tp_ll_time = time.perf_counter() - t0
    save_ckpt('tp_ll_tabular.pt', influence=tp_ll_influence, time_s=tp_ll_time)
    print(f'Traceprop-LL done: {tp_ll_influence.shape}, {tp_ll_time:.2f}s')

# ── Traceprop-BM (batch-mean gradients) ──────────────────────────────────────
if ckpt_exists('tp_bm_tabular.pt'):
    ckpt = load_ckpt('tp_bm_tabular.pt')
    tp_bm_influence = ckpt['influence']
    tp_bm_time = ckpt['time_s']
    print(f'Traceprop-BM loaded: {tp_bm_influence.shape}')
else:
    t0 = time.perf_counter()
    train_grads_bm = batch_mean_gradients(model, X_train, y_train)
    test_grads_bm  = per_sample_gradients(model, X_test,  y_test)   # test always per-sample
    tp_bm_influence = test_grads_bm @ train_grads_bm.T
    tp_bm_time = time.perf_counter() - t0
    save_ckpt('tp_bm_tabular.pt', influence=tp_bm_influence, time_s=tp_bm_time)
    print(f'Traceprop-BM done: {tp_bm_influence.shape}, {tp_bm_time:.2f}s')

  [Loaded: tp_ll_tabular.pt]
Traceprop-LL loaded: (1500, 6000)
  [Loaded: tp_bm_tabular.pt]
Traceprop-BM loaded: (1500, 6000)


## 6. Ground-Truth Retraining (500 subsets)

In [8]:
from sklearn.linear_model import LogisticRegression as SklearnLR

if ckpt_exists('retrain_tabular.pt'):
    ckpt = load_ckpt('retrain_tabular.pt')
    subset_masks   = ckpt['subset_masks']
    subset_margins = ckpt['subset_margins']
    retrain_time   = ckpt['retrain_time']
    print(f'Ground truth loaded: {subset_margins.shape}, took {retrain_time:.1f}s originally')
else:
    print(f'Retraining {N_SUBSETS} logistic regression models (sklearn)...')
    np.random.seed(SEED + 1000)
    all_masks = [np.random.rand(N_TRAIN) < SUBSET_RATIO for _ in range(N_SUBSETS)]
    subset_masks   = np.array(all_masks)
    subset_margins = np.zeros((N_SUBSETS, N_TEST), dtype=np.float32)

    # Use numpy arrays for sklearn (faster than torch tensors)
    X_tr_np = X_train.cpu().numpy()
    y_tr_np = y_train.cpu().numpy()
    X_te_np = X_test.cpu().numpy()
    y_te_np = y_test.cpu().numpy()

    t0 = time.perf_counter()
    for s, mask in enumerate(all_masks):
        clf = SklearnLR(max_iter=200, C=10.0, solver='lbfgs', random_state=s, n_jobs=1)
        clf.fit(X_tr_np[mask], y_tr_np[mask])

        # Margin = logit[correct] - logit[incorrect]
        logits = clf.predict_log_proba(X_te_np)       # (N_TEST, 2), log-probs
        correct = logits[np.arange(N_TEST), y_te_np]
        wrong   = logits[np.arange(N_TEST), 1 - y_te_np]
        subset_margins[s] = (correct - wrong).astype(np.float32)

        if s % 50 == 0 or s == N_SUBSETS - 1:
            elapsed = time.perf_counter() - t0
            eta = elapsed / (s + 1) * (N_SUBSETS - s - 1)
            acc = (subset_margins[s] > 0).mean()
            print(f'  Subset {s+1}/{N_SUBSETS}: acc={acc:.4f} '
                  f'[{elapsed:.0f}s elapsed, ~{eta:.0f}s remaining]')

    retrain_time = time.perf_counter() - t0
    save_ckpt('retrain_tabular.pt',
              subset_masks=subset_masks,
              subset_margins=subset_margins,
              retrain_time=retrain_time)
    print(f'\nDone in {retrain_time:.1f}s ({retrain_time/60:.1f} min)')

Retraining 500 logistic regression models (sklearn)...
  Subset 1/500: acc=0.8267 [1s elapsed, ~368s remaining]


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1458: RuntimeWarning: divide by zero encountered in log
  return np.log(self.predict_proba(X))
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1458: RuntimeWarning: divide by zero encountered in log
  return np.log(self.predict_proba(X))
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1458: RuntimeWarning: divide by zero encountered in log
  return np.log(self.predict_proba(X))
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1458: RuntimeWarning: divide by zero encountered in log
  return np.log(self.predict_proba(X))
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1458: RuntimeWarning: divide by zero encountered in log
  return np.log(self.predict_proba(X))
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1458: RuntimeWarning: divide by zero encountered in log
  return np.log(self.pred

  Subset 51/500: acc=0.8233 [11s elapsed, ~97s remaining]


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1458: RuntimeWarning: divide by zero encountered in log
  return np.log(self.predict_proba(X))
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1458: RuntimeWarning: divide by zero encountered in log
  return np.log(self.predict_proba(X))
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1458: RuntimeWarning: divide by zero encountered in log
  return np.log(self.predict_proba(X))
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1458: RuntimeWarning: divide by zero encountered in log
  return np.log(self.predict_proba(X))
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1458: RuntimeWarning: divide by zero encountered in log
  return np.log(self.predict_proba(X))
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1458: RuntimeWarning: divide by zero encountered in log
  return np.log(self.pred

  Subset 101/500: acc=0.8380 [14s elapsed, ~55s remaining]


...

  Subset 401/500: acc=0.8240 [30s elapsed, ~7s remaining]


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1458: RuntimeWarning: divide by zero encountered in log
  return np.log(self.predict_proba(X))
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1458: RuntimeWarning: divide by zero encountered in log
  return np.log(self.predict_proba(X))
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1458: RuntimeWarning: divide by zero encountered in log
  return np.log(self.predict_proba(X))
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1458: RuntimeWarning: divide by zero encountered in log
  return np.log(self.predict_proba(X))
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1458: RuntimeWarning: divide by zero encountered in log
  return np.log(self.predict_proba(X))
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1458: RuntimeWarning: divide by zero encountered in log
  return np.log(self.pred

  Subset 451/500: acc=0.8287 [35s elapsed, ~4s remaining]


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1458: RuntimeWarning: divide by zero encountered in log
  return np.log(self.predict_proba(X))
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1458: RuntimeWarning: divide by zero encountered in log
  return np.log(self.predict_proba(X))
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1458: RuntimeWarning: divide by zero encountered in log
  return np.log(self.predict_proba(X))
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1458: RuntimeWarning: divide by zero encountered in log
  return np.log(self.predict_proba(X))
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1458: RuntimeWarning: divide by zero encountered in log
  return np.log(self.predict_proba(X))
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1458: RuntimeWarning: divide by zero encountered in log
  return np.log(self.pred

  Subset 500/500: acc=0.8347 [37s elapsed, ~0s remaining]
  [Saved: retrain_tabular.pt]

Done in 37.1s (0.6 min)


## 7. Compute LDS

In [9]:
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)


def compute_lds(influence_matrix, subset_masks, subset_margins, label=''):
    n_test = influence_matrix.shape[0]
    lds = []
    for i in range(n_test):
        predicted = subset_masks @ influence_matrix[i]
        actual    = subset_margins[:, i]
        corr = spearmanr(predicted, actual).statistic
        lds.append(corr if not np.isnan(corr) else 0.0)
        if i % 300 == 0 and label:
            print(f'  [{label}] {i}/{n_test}')
    return np.array(lds)


print('Computing LDS...')

t0 = time.perf_counter()
lds_ll = compute_lds(tp_ll_influence, subset_masks, subset_margins, 'TP-LL')
lds_ll_time = time.perf_counter() - t0
print(f'  Traceprop-LL LDS: {lds_ll.mean():.4f} +/- {lds_ll.std():.4f}')

t0 = time.perf_counter()
lds_bm = compute_lds(tp_bm_influence, subset_masks, subset_margins, 'TP-BM')
lds_bm_time = time.perf_counter() - t0
print(f'  Traceprop-BM LDS: {lds_bm.mean():.4f} +/- {lds_bm.std():.4f}')

np.random.seed(SEED + 2000)
random_matrix = np.random.rand(N_TEST, N_TRAIN)
lds_rand = compute_lds(random_matrix, subset_masks, subset_margins)
print(f'  Random LDS:       {lds_rand.mean():.4f} +/- {lds_rand.std():.4f}')

Computing LDS...
  [TP-LL] 0/1500
  [TP-LL] 300/1500
  [TP-LL] 600/1500
  [TP-LL] 900/1500
  [TP-LL] 1200/1500
  Traceprop-LL LDS: 0.6222 +/- 0.1801
  [TP-BM] 0/1500
  [TP-BM] 300/1500
  [TP-BM] 600/1500
  [TP-BM] 900/1500
  [TP-BM] 1200/1500
  Traceprop-BM LDS: 0.0127 +/- 0.0436
  Random LDS:       -0.0081 +/- 0.0443


## 8. Results

In [10]:
print('\n' + '=' * 65)
print('Adult Income / Logistic Regression — LDS Results')
print('=' * 65)
print(f'{"Method":<42} {"LDS mean":>8} {"± std":>8} {"Time":>8}')
print('-' * 65)
print(f'{"Traceprop-LL (per-sample, exact)":<42} {lds_ll.mean():>8.4f} {lds_ll.std():>8.4f} {tp_ll_time:>7.2f}s')
print(f'{"Traceprop-BM (batch-mean)":<42} {lds_bm.mean():>8.4f} {lds_bm.std():>8.4f} {tp_bm_time:>7.2f}s')
print(f'{"Random baseline":<42} {lds_rand.mean():>8.4f} {lds_rand.std():>8.4f} {"<0.001s":>8}')
print('=' * 65)

results = {
    'benchmark': 'Adult-Income/LogisticRegression',
    'n_train': N_TRAIN,
    'n_test': N_TEST,
    'n_features': N_FEATURES,
    'n_subsets': N_SUBSETS,
    'full_model_accuracy': round(full_acc, 4),
    'traceprop_lastlayer': {
        'lds_mean': round(float(lds_ll.mean()), 4),
        'lds_std':  round(float(lds_ll.std()),  4),
        'time_s':   round(tp_ll_time, 3),
        'method':   'per_sample_exact_gradient_logreg',
    },
    'traceprop_batchmean': {
        'lds_mean': round(float(lds_bm.mean()), 4),
        'lds_std':  round(float(lds_bm.std()),  4),
        'time_s':   round(tp_bm_time, 3),
        'method':   'batch_mean_gradient_logreg',
    },
    'random': {
        'lds_mean': round(float(lds_rand.mean()), 4),
        'lds_std':  round(float(lds_rand.std()),  4),
    },
    'retrain_time_s': round(retrain_time, 1),
    'device': str(DEVICE),
}

with open('tabular_logistic_lds.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nSaved to tabular_logistic_lds.json')
print(json.dumps(results, indent=2))


Adult Income / Logistic Regression — LDS Results
Method                                     LDS mean    ± std     Time
-----------------------------------------------------------------
Traceprop-LL (per-sample, exact)             0.6222   0.1801    0.22s
Traceprop-BM (batch-mean)                    0.0127   0.0436    0.16s
Random baseline                             -0.0081   0.0443  <0.001s

Saved to tabular_logistic_lds.json
{
  "benchmark": "Adult-Income/LogisticRegression",
  "n_train": 6000,
  "n_test": 1500,
  "n_features": 100,
  "n_subsets": 500,
  "full_model_accuracy": 0.8013,
  "traceprop_lastlayer": {
    "lds_mean": 0.6222,
    "lds_std": 0.1801,
    "time_s": 0.22,
    "method": "per_sample_exact_gradient_logreg"
  },
  "traceprop_batchmean": {
    "lds_mean": 0.0127,
    "lds_std": 0.0436,
    "time_s": 0.163,
    "method": "batch_mean_gradient_logreg"
  },
  "random": {
    "lds_mean": -0.0081,
    "lds_std": 0.0443
  },
  "retrain_time_s": 37.1,
  "device": "cpu"
}
